In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [9]:
import kagglehub
import os
import json
from google.colab import userdata

kaggle_auth_json_str = userdata.get('KAGGLE_AUTH_JSON')

if kaggle_auth_json_str:
    try:
        # Create the .kaggle directory if it doesn't exist
        os.makedirs('/root/.kaggle', exist_ok=True)
        # Write the kaggle.json file
        with open('/root/.kaggle/kaggle.json', 'w') as f:
            f.write(kaggle_auth_json_str)
        # Set the file permissions
        os.chmod('/root/.kaggle/kaggle.json', 600)
        print("Kaggle API key configured successfully.")

        # Now you can download the dataset
        path = kagglehub.dataset_download("kaushal2896/english-to-german")
        print("Path to dataset files:", path)

    except json.JSONDecodeError:
        print("Error: Invalid JSON in KAGGLE_AUTH_JSON Colab secret.")
    except Exception as e:
        print(f"An error occurred: {e}")
else:
    print("Kaggle API key not found in Colab secrets. Please add your kaggle.json content as 'KAGGLE_AUTH_JSON'.")

# Copy the downloaded files from the Kaggle cache to the content directory
# Replace 'kaushal2896/english-to-german/versions/1' with the actual path from the previous output
!cp -r /root/.cache/kagglehub/datasets/kaushal2896/english-to-german/versions/1/* /content/
print("Files copied to /content/")

Kaggle API key configured successfully.
Path to dataset files: /kaggle/input/english-to-german
Files copied to /content/


In [22]:
import re
import unicodedata

class TextCleaner:
    def __init__(self, lowercase: bool = True, remove_punctuation: bool = True):
        self.lowercase = lowercase
        self.remove_punctuation = remove_punctuation

    def unicode_to_ascii(self, s: str) -> str:
        """
        Normalize unicode string to ASCII.
        E.g., “Ç” -> “C”, “ñ” -> “n”
        """
        return ''.join(
            c for c in unicodedata.normalize('NFD', s)
            if unicodedata.category(c) != 'Mn'
        )

    def clean_sentence(self, sentence: str) -> str:
        # Convert unicode to ascii
        sentence = self.unicode_to_ascii(sentence)

        # Lowercase if enabled
        if self.lowercase:
            sentence = sentence.lower()

        # Remove unwanted characters (optional punctuation removal)
        if self.remove_punctuation:
            sentence = re.sub(r"[^a-zA-Z0-9]+", " ", sentence)
        else:
            sentence = re.sub(r"\s+", " ", sentence)

        # Remove extra spaces
        sentence = sentence.strip()

        return sentence

    def __call__(self, sentence: str) -> str:
        return self.clean_sentence(sentence)

In [ ]:
from torch.utils.data import Dataset

class TranslationDataset(Dataset):
    def __init__(self, file_path, tokenizer_src, tokenizer_tgt):
        self.pairs = []
        with open(file_path, encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 2:
                    self.pairs.append((parts[0], parts[1]))

        self.tokenizer_src = tokenizer_src
        self.tokenizer_tgt = tokenizer_tgt

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        src_tokens = self.tokenizer_src(src)
        tgt_tokens = self.tokenizer_tgt(tgt)
        return src_tokens, tgt_tokens


In [24]:
class TextProcessor:
    def __init__(self, path= "/content/deu.txt"):
        self.path = path
        self.cleaner = TextCleaner()

    def openFileAndCreateTextPairs(self):
        with open(self.path, "r", encoding="utf-8") as f:
            lines = f.read().strip().split("\n")
        pairs = [[line.split("\t")[0],line.split("\t")[1]] for line in lines]
        return pairs

    def cleanText(self):
        pairs = self.openFileAndCreateTextPairs()
        pairs = [[self.cleaner(pair[0]), self.cleaner(pair[1])] for pair in pairs]
        return pairs

In [25]:
textProcessor = TextProcessor()
pairs = textProcessor.cleanText()
pairs

[['go', 'geh'],
 ['hi', 'hallo'],
 ['hi', 'gru gott'],
 ['run', 'lauf'],
 ['run', 'lauf'],
 ['wow', 'potzdonner'],
 ['wow', 'donnerwetter'],
 ['fire', 'feuer'],
 ['help', 'hilfe'],
 ['help', 'zu hulf'],
 ['stop', 'stopp'],
 ['wait', 'warte'],
 ['wait', 'warte'],
 ['begin', 'fang an'],
 ['go on', 'mach weiter'],
 ['hello', 'hallo'],
 ['hurry', 'beeil dich'],
 ['hurry', 'schnell'],
 ['i hid', 'ich versteckte mich'],
 ['i hid', 'ich habe mich versteckt'],
 ['i ran', 'ich rannte'],
 ['i see', 'ich verstehe'],
 ['i see', 'aha'],
 ['i try', 'ich probiere es'],
 ['i won', 'ich hab gewonnen'],
 ['i won', 'ich habe gewonnen'],
 ['relax', 'entspann dich'],
 ['shoot', 'feuer'],
 ['shoot', 'schie'],
 ['smile', 'lacheln'],
 ['ask me', 'frag mich'],
 ['ask me', 'fragt mich'],
 ['ask me', 'fragen sie mich'],
 ['attack', 'angriff'],
 ['attack', 'attacke'],
 ['cheers', 'zum wohl'],
 ['eat it', 'iss es'],
 ['eat up', 'iss auf'],
 ['eat up', 'iss auf'],
 ['freeze', 'keine bewegung'],
 ['freeze', 'stehenb

[['Go.',
  'Geh.',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #8597805 (Roujin)'],
 ['Hi.',
  'Hallo!',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #380701 (cburgmer)'],
 ['Hi.',
  'Grüß Gott!',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #659813 (Esperantostern)'],
 ['Run!',
  'Lauf!',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #941078 (Fingerhut)'],
 ['Run.',
  'Lauf!',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #4008918 (JSakuragi) & #941078 (Fingerhut)'],
 ['Wow!',
  'Potzdonner!',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #52027 (Zifre) & #2122382 (Pfirsichbaeumchen)'],
 ['Wow!',
  'Donnerwetter!',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #52027 (Zifre) & #2122391 (Pfirsichbaeumchen)'],
 ['Fire!',
  'Feuer!',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #1829639 (Spamster) & #1958697 (Tamy)'],
 ['Help!',
  'Hilfe!',
  'CC-BY 2.0 (France) Attribution: tatoeba.org #435084 (lukaszpp) & #

In [2]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, num_layers, batch_first=True)

    def forward(self, input):
        embedded = self.embedding(input)
        output, hidden = self.gru(embedded)
        return output, hidden

class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size

        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, input, hidden):
        # input: (batch_size,)
        embedded = self.embedding(input).unsqueeze(1)  # (batch_size, 1, hidden_size)
        embedded = F.relu(embedded)
        output, hidden = self.gru(embedded, hidden)  # output: (batch_size, 1, hidden_size)
        output = self.out(output.squeeze(1))  # (batch_size, output_size)
        return output, hidden

class EncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder):
        super(EncoderDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        max_len = tgt.size(1)
        vocab_size = self.decoder.output_size

        outputs = torch.zeros(batch_size, max_len, vocab_size).to(src.device)

        encoder_output, hidden = self.encoder(src)

        decoder_input = tgt[:, 0]

        for t in range(1, max_len):
            output, hidden = self.decoder(decoder_input, hidden)
            outputs[:, t] = output
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            decoder_input = tgt[:, t] if teacher_force else output.argmax(1)

        return outputs

In [ ]:
vocab_size = 10000
input_vocab_size = vocab_size
output_vocab_size = vocab_size
hidden_size = 256
batch_size = 32
src_len = 10
tgt_len = 10

encoder = EncoderRNN(input_vocab_size, hidden_size)
decoder = DecoderRNN(hidden_size, output_vocab_size)
model = EncoderDecoder(encoder, decoder)

src_seq = torch.randint(0, input_vocab_size, (batch_size, src_len))
tgt_seq = torch.randint(0, output_vocab_size, (batch_size, tgt_len))
output = model(src_seq, tgt_seq)
print(f"Output shape: {output.shape}")

Output shape: torch.Size([32, 10, 10000])


In [ ]:
num_epochs = 1000
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

# Training loop
for epoch in range(num_epochs):
    output = model(src_seq, tgt_seq)  # (batch, tgt_len, vocab)
    loss = 0
    for t in range(1, tgt_len):
        loss += criterion(output[:, t, :], tgt_seq[:, t])  # per timestep
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

In [ ]:
def token_accuracy(predictions, targets, pad_token=None):
    """
    predictions: (batch_size, seq_len)
    targets:     (batch_size, seq_len)
    """
    assert predictions.shape == targets.shape

    if pad_token is not None:
        mask = (targets != pad_token)
        correct = (predictions == targets) & mask
        total = mask.sum()
    else:
        correct = (predictions == targets)
        total = torch.numel(targets)

    accuracy = float(correct.sum()) / total
    return accuracy
predictions = output.argmax(dim=-1)
token_accuracy(predictions, tgt_seq)

0.9

In [ ]:
tgt_seq.shape

torch.Size([32, 10])